In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_excel("Online Retail.xlsx")


In [3]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [4]:

# assuming your dataset is already loaded as df

profile = pd.DataFrame({
    "dtype": df.dtypes,
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100,
    "unique_values": df.nunique()
})

# numeric statistics
numeric_stats = df.describe().T[["min", "max", "mean", "50%", "std"]]
numeric_stats = numeric_stats.rename(columns={"50%": "median"})

# combine everything
profile = profile.join(numeric_stats)

# dataset size info
rows, cols = df.shape
memory_usage = df.memory_usage(deep=True).sum()

print(f"Rows: {rows}")
print(f"Columns: {cols}")
print(f"Memory usage: {memory_usage / 1024**2:.2f} MB")

profile

Rows: 541909
Columns: 8
Memory usage: 126.18 MB


,dtype,missing_count,missing_percent,unique_values,min,max,mean,median,std
InvoiceNo,object,0,0.000000,25900,NaN,NaN,NaN,NaN,NaN
StockCode,object,0,0.000000,4070,NaN,NaN,NaN,NaN,NaN
Description,object,1454,0.268311,4223,NaN,NaN,NaN,NaN,NaN
Quantity,int64,0,0.000000,722,-80995.0,80995.0,9.55225,3.0,218.081158
InvoiceDate,datetime64[ns],0,0.000000,23260,2010-12-01 08:26:00,2011-12-09 12:50:00,2011-07-04 13:34:57.156386048,2011-07-19 17:17:00,NaN
UnitPrice,float64,0,0.000000,1630,-11062.06,38970.0,4.611114,2.08,96.759853
CustomerID,float64,135080,24.926694,4372,12346.0,18287.0,15287.69057,15152.0,1713.600303
Country,object,0,0.000000,38,NaN,NaN,NaN,NaN,NaN


In [5]:
missing_customer_df = df[df["CustomerID"].isna()]
with_customer_df = df[df["CustomerID"].notna()]
missing_description_df = df[df["Description"].isna()]
with_description_df = df[df["Description"].notna()]


print(f"Total amout of missing customers: {missing_customer_df.shape[0]}")
print(f"Total amout of missing descriptions: {missing_description_df.shape[0]}\n")


# Count of countries
print(missing_customer_df['Country'].value_counts())
print(with_customer_df['Country'].value_counts())

Total amout of missing customers: 135080
Total amout of missing descriptions: 1454

Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
France                66
Israel                47
Portugal              39
Bahrain                2
Name: count, dtype: int64
Country
United Kingdom          361878
Germany                   9495
France                    8491
EIRE                      7485
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               1877
Portugal                  1480
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
USA                        29

In [6]:
# Summary stats
print(missing_customer_df[['Quantity', 'UnitPrice']].describe())
print(with_customer_df[['Quantity', 'UnitPrice']].describe())

            Quantity      UnitPrice
count  135080.000000  135080.000000
mean        1.995573       8.076577
std        66.696153     151.900816
min     -9600.000000  -11062.060000
25%         1.000000       1.630000
50%         1.000000       3.290000
75%         3.000000       5.450000
max      5568.000000   17836.460000
            Quantity      UnitPrice
count  406829.000000  406829.000000
mean       12.061303       3.460471
std       248.693370      69.315162
min    -80995.000000       0.000000
25%         2.000000       1.250000
50%         5.000000       1.950000
75%        12.000000       3.750000
max     80995.000000   38970.000000


In [7]:
print("Missing value count before cleaning:")
print(df.isna().sum())

df = df[df['CustomerID'].notna()]  # drops all rows where CustomerID is missing

print("\nMissing value count after cleaning:")
print(df.isna().sum())

Missing value count before cleaning:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Missing value count after cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


#### Task 1 deliverable

The column with the most significant amount of missing data was **CustomerID**. After examining the rows with missing values, I observed that the majority of transactions without a `CustomerID` were concentrated in a specific country, namely **the United Kingdom**. Because the missing values were not randomly distributed and appeared to depend on another variable (`Country`), the missingness was classified as **MAR (Missing At Random)**.

Since it is not possible to reliably reconstruct or infer the missing `CustomerID` values, the chosen strategy was **deletion**. All rows with missing `CustomerID` values were removed from the dataset.

After performing this deletion, the dataset was checked again for missing values. It turned out that the other columns that appeared to have missing values (such as `Description`) were associated with the same rows that had missing `CustomerID`. As a result, removing those rows effectively **eliminated all missing values from the dataset in a single step**.

## Task 2

In [8]:
cancellations_df = df[df['InvoiceNo'].str.startswith('C', na=False)]
cancellations_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [9]:
num_cancellations = len(cancellations_df)
print(f"Number of cancellation transactions: {num_cancellations}")

Number of cancellation transactions: 8905


In [10]:
cancellations_df[['Quantity', 'UnitPrice']].describe()

,Quantity,UnitPrice
count,8905.000000,8905.000000
mean,-30.859966,18.845519
std,1170.154939,444.366043
min,-80995.000000,0.010000
25%,-6.000000,1.450000
50%,-2.000000,2.950000
75%,-1.000000,4.950000
max,-1.000000,38970.000000


In [11]:
# Flag cancellations
df['is_cancellation'] = df['InvoiceNo'].str.startswith('C', na=False)


df['is_cancellation'].value_counts()

is_cancellation
False    397924
True       8905
Name: count, dtype: int64

Cancellations represent returns, not purchases, and can distort revenue, churn signals, and recommender systems if treated as normal transactions. Flagging them allows downstream analysis to account for negative behavior without losing data.

In [12]:
# Non-cancellation rows
non_cancels = df[~df['is_cancellation']]

# Quantity <= 0
invalid_quantity = non_cancels[non_cancels['Quantity'] <= 0]

# Count
num_invalid_quantity = len(invalid_quantity)
print(f"Number of non-cancellation transactions with Quantity <= 0: {num_invalid_quantity}")

Number of non-cancellation transactions with Quantity <= 0: 0


In [13]:
invalid_unitprice = df[df['UnitPrice'] <= 0]
num_invalid_unitprice = len(invalid_unitprice)
print(f"Number of transactions with UnitPrice <= 0: {num_invalid_unitprice}")
df.shape

Number of transactions with UnitPrice <= 0: 40


(406829, 9)

In [14]:
df = df[df['UnitPrice'] > 0]
df.shape

(406789, 9)

In [15]:
# Quantity outliers
Q1_qty = df['Quantity'].quantile(0.25)
Q3_qty = df['Quantity'].quantile(0.75)
IQR_qty = Q3_qty - Q1_qty

qty_outliers = df[(df['Quantity'] < (Q1_qty - 1.5*IQR_qty)) | (df['Quantity'] > (Q3_qty + 1.5*IQR_qty))]
print(f"Number of Quantity outliers: {len(qty_outliers)}")

# UnitPrice outliers
Q1_price = df['UnitPrice'].quantile(0.25)
Q3_price = df['UnitPrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price

price_outliers = df[(df['UnitPrice'] < (Q1_price - 1.5*IQR_price)) | (df['UnitPrice'] > (Q3_price + 1.5*IQR_price))]
print(f"Number of UnitPrice outliers: {len(price_outliers)}")

Number of Quantity outliers: 26673
Number of UnitPrice outliers: 36051


In [16]:
df.loc[:, 'Quantity_outlier'] = ((df['Quantity'] < (Q1_qty - 1.5*IQR_qty)) | 
                                 (df['Quantity'] > (Q3_qty + 1.5*IQR_qty)))
df.loc[:, 'UnitPrice_outlier'] = ((df['UnitPrice'] < (Q1_price - 1.5*IQR_price)) |
                                  (df['UnitPrice'] > (Q3_price + 1.5*IQR_price)))

In [17]:
print(df['Quantity_outlier'].sum(), df['UnitPrice_outlier'].sum())

26673 36051


### Handling Invalid Quantities and Prices


1. **Quantity ≤ 0 (non-cancellations):** None found — dataset is clean.  
2. **UnitPrice ≤ 0:** 40 rows — likely data entry errors; these were removed.  
3. **Quantity outliers:** 26,682 rows  
4. **UnitPrice outliers:** 36,051 rows  

For **Quantity** and **UnitPrice outliers**, I chose to **flag them** instead of removing.  

**Reasoning:**  
- Extreme values could be legitimate bulk orders or high-value products.  
- Flagging allows downstream tasks (e.g., recommender systems or churn prediction) to **handle these rows differently** without losing information.  
- Outliers were identified using the **Interquartile Range (IQR) method**, a standard statistical approach.

#### Deliverable for Task 2

1. **Removed rows with UnitPrice ≤ 0** — 40 rows, likely data entry errors.  
2. **Flagged extreme Quantity and UnitPrice values** using the Interquartile Range (IQR) method for outlier detection.  
3. **Flagged cancellations** separately to preserve information on returns.  

**Validation after cleaning:**

- No remaining negative quantities in non-cancellation transactions.  
- No zero or negative UnitPrice values.  
- Dataset shape was verified before and after cleaning to ensure only intended rows were removed.  


## Task 3

In [18]:
# Total number of unique countries
num_unique_countries = df['Country'].nunique()
print(f"Number of unique countries: {num_unique_countries}")

Number of unique countries: 37


In [19]:
# Top 5 countries by transaction count
top_countries = df['Country'].value_counts().head(5)
top_countries_percentage = top_countries.sum() / len(df) * 100
print(f"Top 5 countries account for {top_countries_percentage:.2f}% of transactions\n")

# Count transactions per country
country_counts = df['Country'].value_counts()

# Convert counts to percentage of total transactions
country_percentages = (country_counts / len(df) * 100).round(2)

# Display as a Series: Country name → percentage
top5_countries = country_percentages.head(5)
print(top5_countries)

Top 5 countries account for 95.84% of transactions

Country
United Kingdom    88.95
Germany            2.33
France             2.09
EIRE               1.84
Spain              0.62
Name: count, dtype: float64


In [20]:
# Countries with fewer than 50 transactions
low_activity_countries = (df['Country'].value_counts() < 50).sum()
print(f"Number of countries with fewer than 50 transactions: {low_activity_countries}\n")

# Count transactions per country
country_counts = df['Country'].value_counts()

# Filter countries with fewer than 50 transactions
low_activity_series = country_counts[country_counts < 50]
print(low_activity_series)

Number of countries with fewer than 50 transactions: 6

Country
Lebanon           45
Lithuania         35
Brazil            32
Czech Republic    30
Bahrain           17
Saudi Arabia      10
Name: count, dtype: int64


In [21]:
# Identify rare countries
rare_countries = country_counts[country_counts < 50].index

# Create cleaned Country column
df['Country_cleaned'] = df['Country'].replace(rare_countries, 'Other')

# Compare number of categories before and after
num_before = df['Country'].nunique()
num_after = df['Country_cleaned'].nunique()

print(f"Number of categories before: {num_before}")
print(f"Number of categories after grouping rare countries into 'Other': {num_after}")

df.head()

Number of categories before: 37
Number of categories after grouping rare countries into 'Other': 32


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancellation,Quantity_outlier,UnitPrice_outlier,Country_cleaned
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,False,False,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,False,False,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,False,False,United Kingdom


In [22]:
num_unique_stockcodes = df['StockCode'].nunique()
print(f"Number of unique stock codes: {num_unique_stockcodes}")

Number of unique stock codes: 3684


In [25]:
# Convert to string
stockcode_str = df['StockCode'].astype(str)

# Mask for non-numeric codes
mask_non_numeric = ~stockcode_str.str.isnumeric()

# Unique non-numeric StockCodes
non_numeric_codes = stockcode_str[mask_non_numeric].unique()
print("Non-product / special code count:", len(non_numeric_codes), '\n')
print("Non-product / special codes:", non_numeric_codes)

Non-product / special code count: 886 

Non-product / special codes: ['85123A' '84406B' '84029G' '84029E' 'POST' '82494L' '85099C' '84997B'
 '84997C' '84519A' '85183B' '85071B' '37444A' '37444C' '84971S' '15056BL'
 '15056N' 'D' '35004C' '85049A' '85099B' '35004G' '85014B' '85014A'
 '84970S' '84030E' '35004B' '85049E' '17091A' '84509A' '84510A' '84709B'
 '84625C' '84625A' '47570B' '85049C' '85049D' '85049G' '84970L' '90199C'
 '90129F' '90210B' '72802C' '85169B' '85099F' '85184C' '35591T' '84032B'
 '85049H' '72800E' '84849B' '90200B' '90059B' '90185C' '90059E' '90059C'
 '90200C' '90200D' '90200A' '16258A' '85231B' '85231G' '48173C' '47563A'
 '84558A' '46000M' '71406C' '84985A' '84596E' '84997D' '47599A' '47599B'
 '85035B' '84968C' '72800B' '84563A' '47504H' '17164B' '15044B' '84569B'
 '85114B' '85114C' '85199L' '85199S' '85019A' '85019C' '85071A' '85071C'
 '85135B' '85136A' '85136C' 'C2' '79144B' '46000R' '46000S' '84508A'
 '85232B' '79066K' '84884A' '51014C' '51014L' '51014A' '79302M' '

In [27]:
# Convert to string (prevents errors with NaN / mixed types)
stockcode_str = df['StockCode'].astype(str)

# Keep only numeric StockCodes (i.e., real products)
df = df[stockcode_str.str.isnumeric()].copy()


### Handling Non-Product StockCodes

Non-numeric StockCodes were identified as non-product transactions, such as postage, discounts, or manual adjustments.

For a **product-level analysis**, these records were **removed**.

**Justification:**
- These codes do not represent actual products.
- Including them would distort product-level insights such as purchase frequency and co-occurrence patterns.
- The number of such records is small relative to the dataset, so their removal does not significantly impact the analysis.


In [28]:
# Make Description safe to use
desc = df['Description'].astype(str).str.upper()

# Create keyword flags
df['is_set'] = desc.str.contains('SET')
df['is_pack'] = desc.str.contains('PACK')
df['is_vintage'] = desc.str.contains('VINTAGE')

In [29]:
# Compare UnitPrice
print(df.groupby('is_set')['UnitPrice'].mean())
print(df.groupby('is_pack')['UnitPrice'].mean())
print(df.groupby('is_vintage')['UnitPrice'].mean())

# Compare Quantity
print(df.groupby('is_set')['Quantity'].mean())
print(df.groupby('is_pack')['Quantity'].mean())
print(df.groupby('is_vintage')['Quantity'].mean())

is_set
False    2.913402
True     2.980542
Name: UnitPrice, dtype: float64
is_pack
False    2.991889
True     0.913582
Name: UnitPrice, dtype: float64
is_vintage
False    2.898964
True     3.225508
Name: UnitPrice, dtype: float64
is_set
False    12.300898
True     10.793940
Name: Quantity, dtype: float64
is_pack
False    11.801880
True     21.136769
Name: Quantity, dtype: float64
is_vintage
False    12.110403
True     12.216812
Name: Quantity, dtype: float64


## Task 4

In [30]:
# Create a revenue column first
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Aggregate per customer
customer_df = df.groupby("CustomerID").agg(
    total_revenue = ("Revenue", "sum"),
    num_orders = ("InvoiceNo", "nunique"),
    num_products = ("StockCode", "nunique"),
    first_purchase = ("InvoiceDate", "min"),
    last_purchase = ("InvoiceDate", "max")
).reset_index()

In [31]:
# Compute 90th percentile threshold
threshold = customer_df['total_revenue'].quantile(0.9)

# Create binary target
customer_df['high_value'] = (customer_df['total_revenue'] >= threshold).astype(int)

In [37]:
# Count of each class
class_counts = customer_df['high_value'].value_counts()
print(class_counts)

# Percentage of each class
class_percent = customer_df['high_value'].value_counts(normalize=True) * 100
print(class_percent)

high_value
0    3906
1     435
Name: count, dtype: int64
high_value
0    89.979267
1    10.020733
Name: proportion, dtype: float64


In [38]:
# Accuracy if model always predicts 0
accuracy_always_regular = (customer_df['high_value'] == 0).mean()
print(f"Accuracy if always predict 'regular': {accuracy_always_regular:.2f}")

Accuracy if always predict 'regular': 0.90


### Class Imbalance Analysis

- **Class distribution:**
  - High-value (1): ~10%
  - Regular (0): ~90%

- **Baseline accuracy:**
  - A naive model always predicting "regular" would achieve ~90% accuracy.

- **Why accuracy is misleading:**
  - High accuracy is dominated by the majority class.
  - Such a model would **completely fail to identify high-value customers**, which is the primary interest.
  - Metrics like **Precision, Recall, F1-score, or ROC-AUC** are more appropriate for evaluating performance on imbalanced datasets.

In [39]:
from sklearn.model_selection import train_test_split

# Features (you can select your numerical features, e.g., total_revenue, num_orders, num_products)
X = customer_df[['total_revenue', 'num_orders', 'num_products']]
y = customer_df['high_value']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [40]:
print("Original training class distribution:")
print(y_train.value_counts())

Original training class distribution:
high_value
0    3124
1     348
Name: count, dtype: int64


In [41]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Random oversampling
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

# Random undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

# Print class distribution after resampling
print("Oversampled class distribution:")
print(y_train_over.value_counts())

print("Undersampled class distribution:")
print(y_train_under.value_counts())

Oversampled class distribution:
high_value
0    3124
1    3124
Name: count, dtype: int64
Undersampled class distribution:
high_value
0    348
1    348
Name: count, dtype: int64


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Function to train and evaluate
def train_evaluate(X_tr, y_tr, X_te, y_te):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    precision = precision_score(y_te, y_pred)
    recall = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    return precision, recall, f1

# Original training set
precision_orig, recall_orig, f1_orig = train_evaluate(X_train, y_train, X_test, y_test)

# Oversampled training set
precision_over, recall_over, f1_over = train_evaluate(X_train_over, y_train_over, X_test, y_test)

# Undersampled training set
precision_under, recall_under, f1_under = train_evaluate(X_train_under, y_train_under, X_test, y_test)

# Print results
print("Original training set - Precision: {:.2f}, Recall: {:.2f}, F1: {:.2f}".format(precision_orig, recall_orig, f1_orig))
print("Oversampled training set - Precision: {:.2f}, Recall: {:.2f}, F1: {:.2f}".format(precision_over, recall_over, f1_over))
print("Undersampled training set - Precision: {:.2f}, Recall: {:.2f}, F1: {:.2f}".format(precision_under, recall_under, f1_under))

Original training set - Precision: 1.00, Recall: 1.00, F1: 1.00
Oversampled training set - Precision: 1.00, Recall: 1.00, F1: 1.00
Undersampled training set - Precision: 0.97, Recall: 1.00, F1: 0.98


## Task 5

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Features & target
X = customer_df[['total_revenue', 'num_orders', 'num_products']]
y = customer_df['high_value']  # Or some target you defined

# Random train/test split (WRONG: ignores time)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train simple model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict & evaluate
y_pred = model.predict(X_test)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

Precision: 1.00, Recall: 1.00, F1: 1.00


### Temporal Leakage Example (Intentionally Wrong)

- Features were computed using all transactions from Dec 2010 through Dec 2011.
- Random train/test split was applied.
- A model trained on this split showed artificially high performance.

**Problem:**  
- This approach allows the model to **“see the future”** (Dec 2011 data) during training.
- It inflates metrics and does not reflect real predictive power.
- Temporal leakage can occur whenever features include information from the target period.

**Key takeaway:**  
- Always compute features **only from data prior to the prediction period** when doing time-based predictions.

In [44]:
# Inspect min/max dates used in feature computation
print(customer_df[['first_purchase', 'last_purchase']].describe())

                      first_purchase                  last_purchase
count                           4341                           4341
mean   2011-04-28 12:13:11.969592320  2011-09-08 13:28:43.980649472
min              2010-12-01 08:26:00            2010-12-01 09:53:00
25%              2011-01-12 13:27:00            2011-07-19 09:49:00
50%              2011-03-31 19:54:00            2011-10-20 14:52:00
75%              2011-08-17 15:00:00            2011-11-22 17:32:00
max              2011-12-09 12:16:00            2011-12-09 12:50:00


In [45]:
# Model evaluation metrics
print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

Precision: 1.00, Recall: 1.00, F1: 1.00


In [46]:
feature_corrs = X_train.join(y_train).corr()['high_value'].sort_values(ascending=False)
print(feature_corrs)

high_value       1.000000
num_orders       0.556370
num_products     0.488464
total_revenue    0.381266
Name: high_value, dtype: float64


### Detecting Temporal Leakage

1. **Overlapping time periods:**
   - Features were aggregated using all transactions, including Dec 2011.
   - This means train and test sets share information from the target period.

2. **Suspiciously high performance:**
   - Model metrics such as F1, precision, or recall were unrealistically high.
   - Indicates the model “cheated” by using future information.

3. **Feature-target correlations:**
   - Features computed from the full date range (e.g., total revenue, number of orders) had extremely high correlation with the target.
   - In a real scenario, such high correlation would not occur before the prediction period.

**Conclusion:**  
- High correlations and performance, combined with overlapping time windows, are classic signs of temporal leakage.
- Always compute features only from data **prior to the prediction period** to avoid this.

In [47]:

# Observation window: Dec 2010 → Sep 2011
observation_end = pd.Timestamp("2011-09-30")

# Prediction window: Oct 2011 → Dec 2011
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

In [49]:
# Compute features using df_obs only
df_obs = df[df["InvoiceDate"] <= observation_end].copy()
df_obs['Revenue'] = df_obs['Quantity'] * df_obs['UnitPrice']

customer_features = df_obs.groupby("CustomerID").agg(
    total_revenue = ("Revenue", "sum"),
    num_orders = ("InvoiceNo", "nunique"),
    num_products = ("StockCode", "nunique"),
    first_purchase = ("InvoiceDate", "min"),
    last_purchase = ("InvoiceDate", "max")
).reset_index()

In [50]:
# Check if customer made at least one purchase in prediction window
df_pred_customers = df_pred.groupby("CustomerID")["InvoiceNo"].nunique().reset_index()
df_pred_customers['purchase_in_pred_window'] = 1  # all these made a purchase

# Merge with features, fill 0 for customers who didn't purchase
customer_features = customer_features.merge(
    df_pred_customers[['CustomerID','purchase_in_pred_window']],
    on='CustomerID',
    how='left'
)

customer_features['purchase_in_pred_window'] = customer_features['purchase_in_pred_window'].fillna(0).astype(int)

In [51]:
from sklearn.model_selection import train_test_split

X = customer_features[['total_revenue', 'num_orders', 'num_products']]
y = customer_features['purchase_in_pred_window']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [52]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

Precision: 0.72, Recall: 0.57, F1: 0.64


### Temporal Prediction (Correct Approach)

- **Observation window:** Dec 2010 → Sep 2011  
- **Prediction window:** Oct 2011 → Dec 2011  

**Process:**
1. Features were computed only using transactions in the observation window.  
2. The target variable was defined as whether a customer made at least one purchase in the prediction window.  
3. A logistic regression model was trained on the training set and evaluated on the test set.

**Outcome:**
- Performance metrics (precision, recall, F1) are **lower than the random-split “leakage” scenario**, reflecting a realistic predictive task.  
- This approach **avoids temporal leakage** by ensuring that features do not include information from the target period.